Precision vs Recall for fruad detection system using SMOTE with Logistic Regression and Random Forest Classifier

- SMOTE = Synthetic Minority Oversampling Technique

Dataset
    ->
Train/Test Split
    ->
SMOTE (Train Only)
    ->
Scaling
    ->
Model Training
    ->
Hyperparameter Tuning
    ->
Evaluation

In [1]:
import pandas as pd
df = pd.read_csv("../datasets/week2.csv")

In [2]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
df.shape

(284807, 31)

In [4]:
df["Class"].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [5]:
from sklearn.model_selection import train_test_split

x = df.drop("Class", axis=1)
y = df["Class"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [6]:
x_train.size

6835350

In [7]:
y_train.size

227845

In [8]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
x_train_s, y_train_s = smote.fit_resample(x_train, y_train)

In [9]:
x_train_s.size

13647060

In [10]:
y_train_s.size

454902

In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train_s)
x_test = scaler.transform(x_test)

# Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000)
lr.fit(x_train_s, y_train_s)

LogisticRegression(max_iter=1000)

In [13]:
lr_pred = lr.predict(x_test)

In [14]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score

print("Precision:", precision_score(y_test, lr_pred))

print("Recall:", recall_score(y_test, lr_pred))

print("ROC AUC:", roc_auc_score(y_test, lr_pred))

Precision: 0.1624548736462094
Recall: 0.9183673469387755
ROC AUC: 0.95510376350878


# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train_s, y_train_s)

In [ ]:
rf_pred = rf.predict(x_test)
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("ROC AUC:", roc_auc_score(y_test, rf_pred))

# Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
params = {"n_estimators": [100, 200], "max_depth": [5, 10, None]}
grid = GridSearchCV(RandomForestClassifier(), params, cv=5, scoring="recall")
grid.fit(x_train_s, y_train_s)
print(grid.best_params_)

In [ ]:
# we can also use as the pipeline is mentioned in slides rather than applying everything seperately on data
from imblearn.pipeline import Pipeline

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("smote", SMOTE()),
        ("classifier", RandomForestClassifier()),
    ]
)

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)